# Scalar reaction–advection–diffusion methods

Compare Galerkin RAD, conservative SUPG and UNUSUAL reaction–diffusion
stabilization. The analytical field is $u=1+x+2y$; RAD uses
$\beta=(1/4,-1/2)$ and reaction $c=1/2$. Its independently supplied source is
$f=-3/4+(1/2)u$. UNUSUAL has zero advection and source $f=cu$.

The RAD interface multiplier is the half-advection Robin quantity
$(-\nabla u+\beta u/2)\cdot n$. The small scalar patch errors are separate
from conservation or space-time convergence. Heat and adaptive transport are
covered in the numbered notebooks in this folder.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/6c88ea65e95d95fa7f1c266053c314f1f3d60505c44e2c1902f84328d8bcfbbd/introductory_methods-companion.zip"
COMPANION_SHA256 = "6c88ea65e95d95fa7f1c266053c314f1f3d60505c44e2c1902f84328d8bcfbbd"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("transport/introductory_methods.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)

import numpy as np


## Declare and solve Galerkin RAD equations

The native primary example consumes the conservative skew Galerkin forms
below, with $\beta=(1/4,-1/2)$, $c=1/2$, $u=1+x+2y$ and independently
$f=-3/4+(1/2)u$. The retained constant is a coarse basis, not a local kernel.

$$
\begin{aligned}
a_T(u,v)&=(\nabla u,\nabla v)_T+c(u,v)_T\\
 &\quad +\tfrac12\big[(\beta\cdot\nabla u,v)_T
 -(u,\beta\cdot\nabla v)_T\big],\\
B_T\lambda&=\langle\lambda,v\rangle_{\partial T},\\
C_Tu&=-\langle u,\mu\rangle_{\partial T}.
\end{aligned}
$$

The P1 trace is the half-advection Robin quantity
$\lambda=(-\nabla u+\beta u/2)\cdot n$, with the fixed macroface normal.
The global RHS is explicitly the negative Dirichlet trace functional.
SUPG and UNUSUAL remain separate compatibility comparisons below.


In [ ]:
import importlib.util
from pymhm import Equation, LocalEquations, MultiscaleProblem, assemble, columns, rows, compile_form
from pymhm.meshes.triangle import TriangleMesh
from pymhm.fem.traces.interval import FaceSpace, SkeletonSpace
from pymhm.fem.scalar.operators import boundary_data
from examples.variational_darcy import NativeFormProvider

native_available = importlib.util.find_spec("dolfinx") is not None
if native_available:
    import ufl
    from dolfinx import fem

    mesh = TriangleMesh.unit_square()
    trace_space = SkeletonSpace(mesh, tuple(FaceSpace.uniform(1) for _ in mesh.faces))

    def exact_scalar(points):
        """Return the affine analytical scalar field."""
        return 1 + points[:, 0] + 2 * points[:, 1]

    def local_forms(context):
        """Write Galerkin RAD's skew form and its half-advection Robin traces."""
        space = fem.functionspace(context.space.mesh, ("Lagrange", 2))
        u, v = ufl.TrialFunction(space), ufl.TestFunction(space)
        dx = ufl.dx(domain=context.space.mesh)
        x = ufl.SpatialCoordinate(context.space.mesh)
        trace_basis = []
        for endpoints in context.face_endpoints:
            start, end = endpoints
            tangent = end - start
            t = sum(float(tangent[j]) * (x[j] - float(start[j])) for j in range(2))
            t /= float(tangent @ tangent)
            trace_basis.append((1, 2 * t - 1))
        beta = ufl.as_vector((0.25, -0.5))
        exact = 1 + x[0] + 2 * x[1]
        force = -0.75 + 0.5 * exact
        points = space.tabulate_dof_coordinates()[:, :2].copy()
        return LocalEquations(
            (
                ufl.inner(ufl.grad(u), ufl.grad(v))
                + 0.5 * (ufl.dot(beta, ufl.grad(u)) * v - u * ufl.dot(beta, ufl.grad(v)))
                + 0.5 * u * v
            )
            * dx,
            force * v * dx,
            columns(
                *(
                    float(sign) * basis * v * context.ds(side + 1)
                    for side, sign in enumerate(context.signs)
                    for basis in trace_basis[side]
                )
            ),
            rows(
                *(
                    -float(sign) * basis * u * context.ds(side + 1)
                    for side, sign in enumerate(context.signs)
                    for basis in trace_basis[side]
                )
            ),
            context.trace_dofs,
            coarse_basis=np.ones((len(points), 1)),
            moments=columns(v * dx),
            metadata={"points": points},
        )

    boundary, _ = boundary_data(trace_space, exact_scalar)
    problem = MultiscaleProblem(
        Equation(0, -np.r_[boundary, np.zeros(len(mesh.cells))]),
        NativeFormProvider(mesh, trace_space, local_forms, 2),
        range(len(mesh.cells)),
        trace_space.size,
        (1,) * len(mesh.cells),
    )
    system = assemble(problem)
    solution = system.solve()
    errors = [
        float(np.max(np.abs(field - exact_scalar(record["points"]))))
        for field, record in zip(solution.fields, system.local_metadata, strict=True)
    ]
    assert max(errors) < 1e-10
    print(
        {
            "native_available": True,
            "spaces": "P2 scalar / P1 Robin trace / retained constant",
            "maximum_scalar_nodal_error_by_cell": errors,
            "global_original_relative_residual": solution.raw_residual,
        }
    )
else:
    print(
        {
            "native_available": False,
            "status": "User-written UFL primary solve not executed: select the Pixi fem kernel.",
        }
    )


## Compare the supported methods
These comparisons invoke the existing formulation implementations through the application catalogue and preserve their numerical records. The user-defined primary UFL solve above exercises the generic local/global API; the catalogue drivers remain compatibility comparisons.


In [ ]:
from examples.tutorial_scalar_variants import (
    VARIANTS,
    affine_pressure,
    affine_flux,
    run_variant,
)

selected_methods = ("rad", "rad-supg", "rad-unusual")
records = [run_variant(name) for name in selected_methods]
for record in records:
    print(record["variant"], record["spaces_and_conventions"], record["metrics"])
